# Moving car example by combining Speos files

This tutorial demonstrates how to run moving car workflow use case.
## Prerequisites

### Perform imports

In [1]:
import os
from pathlib import Path

from ansys.speos.core import Part, Speos, launcher
from ansys.speos.core.kernel.client import (
    default_docker_channel,
)
from ansys.speos.core.sensor import SensorCamera
from ansys.speos.core.simulation import SimulationInverse
from ansys.speos.core.source import SourceLuminaire
from ansys.speos.core.workflow.combine_speos import SpeosFileInstance, combine_speos


### Define constants
Constants help ensure consistency and avoid repetition throughout the example.

In [2]:
HOSTNAME = "localhost"
GRPC_PORT = 50098  # Be sure the Speos GRPC Server has been started on this port.
CAR_NAMES = ["BlueCar", "RedCar"]
ENVIRONMENT_NAME = "Env_Simplified"
USE_DOCKER = True  # Set to False if you're running this example locally as a Notebook.
USE_GPU = False

## Coordinate systems

Define the global coordinate systems for each of the assets.

In [3]:
GLOBAL_CS = [
    0,  # Origin x
    0,  # Origin y
    0,  # Origin z
    1,  # x-direction x
    0,  # x-direction y
    0,  # x-direction z
    0,  # y-direction x
    1,  # y-direction y
    0,  # y-direction z
    0,  # z-direction x
    0,  # z-direction y
    1,  # z-direction z
]
CAR_CS = {
    "red": [-4000, 0, 48000, 1.0, 0.0, 0.0, 0.0, 0.0, -1.0, 0.0, 1.0, 0.0],
    "blue": [2000, 0, 35000, 0.0, 0.0, -1.0, -1.0, 0.0, 0.0, 0.0, 1.0, 0.0],
}

## Load assets
Assets used to run this example are available in the
[PySpeos repository](https://github.com/ansys/pyspeos/) on GitHub.

> **Note:** Make sure you
> have downloaded simulation assets and set ``assets_data_path``
> to point to the assets folder.

In [4]:
if USE_DOCKER:  # Running on the remote server.
    assets_data_path = Path("/app") / "assets"
else:
    assets_data_path = Path("/path/to/your/download/assets/directory")

In [5]:
# ## Create connection with speos rpc server
if USE_DOCKER:
    speos = Speos(channel=default_docker_channel())
else:
    speos = launcher.launch_local_speos_rpc_server(port=GRPC_PORT)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.13/site-packages/ansys/tools/common/cyberchannel.py:188: UserWarning: Starting gRPC client without TLS on localhost:50098. This is INSECURE. Consider using a secure connection.
  warn(f"Starting gRPC client without TLS on {target}. This is INSECURE. Consider using a secure connection.")


## Combine several speos files into one project

Here we are building a project with:
- An environment which is a road
- A blue car
- A red car

In [6]:
full_env_path = assets_data_path / f"{ENVIRONMENT_NAME}.speos" / f"{ENVIRONMENT_NAME}.speos"
car_paths = [assets_data_path / f"{car}.speos" / f"{car}.speos" for car in CAR_NAMES]
assets = [
    SpeosFileInstance(
        speos_file=str(full_env_path),
        axis_system=GLOBAL_CS,
    ),
    SpeosFileInstance(
        speos_file=str(car_paths[0]),
        axis_system=CAR_CS["red"],
    ),
    SpeosFileInstance(
        speos_file=str(car_paths[1]),
        axis_system=CAR_CS["blue"],
    ),
]
p = combine_speos(
    speos=speos,
    speos_to_combine=assets,
)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.13/site-packages/ansys/speos/core/project.py:811: UserWarning: The pySpeos feature : FaceStub.read_batch needs a Speos Version of 2025 R2 SP0 or higher.
  f_data_list = face_db.read_batch(refs=f_links)


In [7]:
print(p)

{
    "part_guid": "0afebdd0-a18b-4cb0-bb91-b390aab1affc",
    "materials": [
        {
            "name": "Env_Simplified.Material.1",
            "metadata": {
                "UniqueId": "358e4192-d981-42fa-b97b-dc1cbac762f8"
            },
            "vop_guid": "125ace17-9ef5-4a72-b203-d6df5a7395aa",
            "geometries": {
                "geo_paths": [
                    "Env_Simplified/Sidewalk:530980414"
                ]
            },
            "sop_guid": "4de7056e-9562-4f80-9447-449fa85d1985",
            "description": "",
            "sop_guids": [],
            "vop": {
                "opaque": {},
                "name": "",
                "description": "",
                "metadata": {}
            },
            "sop": {
                "library": {
                    "sop_file_uri": "/app/assets/Env_Simplified.speos/Side Walk_33b4-3fff-6c8d-7671..scattering"
                },
                "name": "",
                "description": "",
              

## Preview the project

User can review the created/loaded project using preview method.

In [8]:
p.preview()

EmbeddableWidget(value='<iframe srcdoc="<!DOCTYPE html>\n<html lang=&quot;en-US&quot; dir=&quot;ltr&quot;>\n  …

## Complete the project with sensor/source/simulation

We are adding a camera sensor to have output results, a luminaire to have a light source.

And, we gather the source and the sensor into a simulation (we will compute it just after).

### Create a sensor

In [9]:
ssr = p.create_sensor(name="Camera.1", feature_type=SensorCamera)
ssr.distortion_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraDistortion_190deg.OPTDistortion"
)
ssr.set_mode_photometric().transmittance_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraTransmittance.spectrum"
)
color_mode = ssr.set_mode_photometric().set_mode_color()
color_mode.red_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityRed.spectrum"
)
color_mode.blue_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityBlue.spectrum"
)
color_mode.green_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityGreen.spectrum"
)

In [10]:
ssr.axis_system = [-2000, 1500, 11000, -1, 0, 0, 0, 1, 0, 0, 0, -1]
ssr.commit()

### Create a source

In this example, a luminaire source is created with an IES file.

More details on creating/editing source examples can be found in core examples.

In [11]:
src = p.create_source(name="Luminaire.1", feature_type=SourceLuminaire)
src.set_intensity_file_uri(
    uri=str(assets_data_path / "IES_C_DETECTOR.ies")
).set_spectrum().set_daylightfluorescent()
src.set_axis_system([0, 10000, 50000, 1, 0, 0, 0, 1, 0, 0, 0, 1])

src.commit()

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.13/site-packages/ansys/speos/core/project.py:230: UserWarning: The pySpeos feature : SourceLuminaire needs a Speos Version of 2025 R2 SP0 or higher.
  feature = SourceLuminaire(


### Create a simulation

More details on creating/editing simulation examples can be found in core examples.

In [12]:
sim = p.create_simulation(name="Inverse.1", feature_type=SimulationInverse)
sim.set_sensor_paths(["Camera.1"]).set_source_paths(["Luminaire.1"])
sim.commit()

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.13/site-packages/ansys/speos/core/project.py:315: UserWarning: The pySpeos feature : SimulationInverse needs a Speos Version of 2025 R2 SP0 or higher.
  feature = SimulationInverse(


## Run the simulation

Simulation can be run using CPU via compute_CPU method or using GPU via compute_GPU method.

In [13]:
run_sim = sim.compute_GPU if USE_GPU else sim.compute_CPU
run_sim()  # Run the simulation

[upload_response {
  info {
    uri: "94648415-7696-4ef8-b9e4-241be52582d8"
    file_name: "Camera.1.Irradiance.xmp"
    file_size: 10641177
  }
  upload_duration {
    nanos: 11262601
  }
}
, upload_response {
  info {
    uri: "1fa3c749-2a73-441e-a2e2-19505206e5e7"
    file_name: "Camera.1.hdr"
    file_size: 518354
  }
  upload_duration {
    nanos: 919228
  }
}
, upload_response {
  info {
    uri: "4f557979-949f-4ad5-933a-b1b57516b5c7"
    file_name: "Camera.1.xmp"
    file_size: 1807032
  }
  upload_duration {
    nanos: 2117403
  }
}
, upload_response {
  info {
    uri: "b43b1b74-6f60-459f-af19-89c7429b5a04"
    file_name: "Camera.1.png"
    file_size: 741155
  }
  upload_duration {
    nanos: 1202515
  }
}
, upload_response {
  info {
    uri: "c4b27d2e-8039-4dcc-a139-06a42e52852f"
    file_name: "Inverse.1.html"
    file_size: 997097
  }
  upload_duration {
    nanos: 1438534
  }
}
]

## Check and review result

Open result (only windows)

In [14]:
if os.name == "nt":
    from ansys.speos.core.workflow.open_result import open_result_image

    open_result_image(simulation_feature=sim, result_name="Camera.1.png")

## Modify part

Move the part via changing the axis_system of a part.

axis_system is a list of 12 float values:
x, y, z,
x_vect_x, x_vect_y, x_vect_z,
y_vect_x, y_vect_y, y_vect_z,
z_vect_x, z_vect_y, z_vect_z.

In [15]:
blue_car_sub_part = p.find(name="BlueCar", feature_type=Part.SubPart)[0]
blue_car_sub_part.set_axis_system([2000, 0.0, 20000, 0.0, 0.0, -1.0, -1.0, 0.0, 0.0, 0.0, 1.0, 0.0])
blue_car_sub_part.commit()

## Re-run simulation with the modified part position

In [16]:
run_sim()

[upload_response {
  info {
    uri: "8f537868-fbc4-44e6-8656-594dd9af3469"
    file_name: "Camera.1.Irradiance.xmp"
    file_size: 10373702
  }
  upload_duration {
    nanos: 10504943
  }
}
, upload_response {
  info {
    uri: "0607f2f0-3db5-41b3-89e6-b3f006b401ee"
    file_name: "Camera.1.hdr"
    file_size: 520007
  }
  upload_duration {
    nanos: 835071
  }
}
, upload_response {
  info {
    uri: "56870fd6-1f91-4459-abe7-88433ade339b"
    file_name: "Camera.1.xmp"
    file_size: 1667182
  }
  upload_duration {
    nanos: 1942344
  }
}
, upload_response {
  info {
    uri: "8e9efdde-a4c8-461d-9ad1-6e1614e78da7"
    file_name: "Camera.1.png"
    file_size: 728272
  }
  upload_duration {
    nanos: 911268
  }
}
, upload_response {
  info {
    uri: "10771db7-1415-4889-8142-127ea9ab64a5"
    file_name: "Inverse.1.html"
    file_size: 981496
  }
  upload_duration {
    nanos: 1149748
  }
}
]

Review result:

In [17]:
if os.name == "nt":
    open_result_image(simulation_feature=sim, result_name="Camera.1.png")

## Modify camera property

Modify the camera, e.g. focal length to 10

In [18]:
cam1 = p.find(name="Camera.1", feature_type=SensorCamera)[0]
cam1.focal_length = 10
cam1.commit()

Re-run the simulation and review result

In [19]:
run_sim()
if os.name == "nt":
    open_result_image(simulation_feature=sim, result_name="Camera.1.png")

In [20]:
speos.close()

True